# Qwen3.8-27B（现代多模态深度思考 Web 工作台）

- **支持显卡**：A100 (40GB / 80GB)、H100 (80GB)、RTX 6000 (96GB)
- **底层引擎**：原生 `vLLM 0.32.0`（80GB 显卡加载 [`Qwen/Qwen3.8-27B`](https://huggingface.co/Qwen/Qwen3.8-27B) BF16，40GB 显卡加载 FP8；`262144` 上下文 + MTP 投机解码）
- **单页全能能力**：多轮图像/视频多模态对话、实时可折叠 `<think>` 深度思考流、全采样超参调节与机器资源实时监控
- **访问方式**：使用 Google Colab 官方内置端口代理（`proxyPort`）在浏览器直接打开工作台，绑定当前 Google 账号鉴权，无需外部隧道或密码，不开放外部 API
- **执行顺序**：`1. 环境检测` → `2. 安装 vLLM 推理引擎与 Web 工作台` → `3. 下载并校验 Qwen3.8-27B 模型权重` → `4. 启动多模态对话 Web 工作台`

### 1. 环境检测
- **功能**：检测 GPU 型号、显存容量、BF16 支持、系统内存、磁盘空间及 PyTorch / CUDA 版本。
- **要求**：支持 `A100`、`H100`、`RTX 6000`，显存 40GB 及以上。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 1. Environment Check {display-mode: "form"}
import os, shutil, subprocess, torch

MIN_VRAM_GIB = 35
REQUIREMENT = "40GB+ VRAM (A100 40GB/80GB, H100 80GB, RTX 6000 96GB)"

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Select Runtime -> Change runtime type -> A100 / H100 / RTX 6000 GPU.")

print(subprocess.run(["nvidia-smi"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True).stdout)

gpu = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
ram_gib = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
disk_gib = shutil.disk_usage("/content").free / 1024**3
bf16 = torch.cuda.is_bf16_supported()
print(f"GPU: {gpu} | VRAM: {vram_gib:.1f} GiB | RAM: {ram_gib:.1f} GiB | Disk Free: {disk_gib:.1f} GiB | BF16: {bf16}")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

if not any(k in gpu.upper() for k in ("A100", "H100", "6000")) or vram_gib < MIN_VRAM_GIB:
    raise RuntimeError(f"Detected {gpu} ({vram_gib:.1f} GiB), but this notebook requires {REQUIREMENT}. Please switch runtime.")
if not bf16:
    raise RuntimeError("Current GPU does not support BF16; please switch runtime.")
if vram_gib < 70:
    print("[Note] 40GB-class GPU detected: Step 3 will automatically select Qwen3.8-27B-FP8.")
print(f"[OK] GPU environment ready: {gpu} ({vram_gib:.1f} GiB)")

### 2. 安装 vLLM 推理引擎与 Web 工作台
- **功能**：安装 vLLM `0.32.0`、`hf_xet` 与 Web 工作台依赖，并拉取校验 GitHub Release 发布的 `app-qwen3.8-27b.tar.gz`（`v0.2.4`，内含多模态对话前端工作台与本地后端）。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 2. Install vLLM Engine & App Bundle {display-mode: "form"}
import hashlib, shutil, subprocess, sys, tarfile, tempfile, urllib.request
from pathlib import Path

VLLM_VERSION = "0.32.0"

APP_DIR = Path("/content/app")
RELEASE_REPO = "boltguo/colab"
RELEASE_TAG = "v0.2.4"
APP_ARCHIVE = "app-qwen3.8-27b.tar.gz"
APP_SHA256 = "ccd142ec6725425806e789627a5c1f446cf20404514cc6942498ff6207fd9ef8"

APP_SERVER_PACKAGES = [
    f"vllm=={VLLM_VERSION}",
    "fastapi>=0.115.0,<1.0.0",
    "uvicorn[standard]>=0.30.0,<1.0.0",
    "python-multipart>=0.0.9,<1.0.0",
    "pydantic>=2.7.0,<3.0.0",
    "huggingface_hub>=0.23.0",
    "hf_xet",
    "openai",
]

def run(args, cwd=None):
    subprocess.run(args, cwd=cwd, check=True)

def ensure_not_symlink(path, label):
    for p in [path, *path.parents]:
        if p == Path("/content"):
            break
        if p.is_symlink():
            raise RuntimeError(f"{label} path contains a symlink; start a fresh runtime: {p}")

def install_release_app():
    ensure_not_symlink(APP_DIR, "App directory")
    stamp_file = APP_DIR / ".release_sha256"
    if (
        stamp_file.is_file()
        and not stamp_file.is_symlink()
        and stamp_file.read_text(encoding="utf-8").strip() == f"{RELEASE_TAG}:{APP_SHA256}"
        and (APP_DIR / "backend/run_server.py").is_file()
    ):
        print(f"App bundle already up to date: {APP_ARCHIVE} ({RELEASE_TAG})")
        return

    local_archive = Path(f"/content/{APP_ARCHIVE}")
    with tempfile.NamedTemporaryFile(suffix=".tar.gz", delete=False) as tmp:
        tmp_path = Path(tmp.name)
    try:
        if local_archive.is_file() and not local_archive.is_symlink():
            print(f"Using pre-staged app archive: {local_archive}")
            payload = local_archive.read_bytes()
        else:
            url = f"https://github.com/{RELEASE_REPO}/releases/download/{RELEASE_TAG}/{APP_ARCHIVE}"
            print(f"Downloading app bundle: {url}")
            req = urllib.request.Request(url, headers={"User-Agent": "colab-release-fetcher"})
            with urllib.request.urlopen(req, timeout=60) as resp:
                payload = resp.read()
        actual_sha = hashlib.sha256(payload).hexdigest()
        if actual_sha != APP_SHA256:
            raise RuntimeError(
                f"App bundle SHA256 mismatch! Expected {APP_SHA256}, got {actual_sha}"
            )
        tmp_path.write_bytes(payload)

        staging = APP_DIR.with_name("app.staging")
        if staging.is_symlink():
            raise RuntimeError("Staging directory is a symlink; aborting.")
        if staging.exists():
            shutil.rmtree(staging)
        staging.mkdir(parents=True, exist_ok=True)
        staging_root = staging.resolve()

        with tarfile.open(tmp_path, "r:gz") as tf:
            for member in tf.getmembers():
                if member.issym() or member.islnk():
                    raise RuntimeError(f"Refusing symlink/hardlink archive entry: {member.name}")
                target = (staging / member.name).resolve()
                if target != staging_root and not str(target).startswith(str(staging_root) + "/"):
                    raise RuntimeError(f"Archive path escapes staging root: {member.name}")
            tf.extractall(staging)

        (staging / ".release_sha256").write_text(f"{RELEASE_TAG}:{APP_SHA256}\n", encoding="utf-8")
        if APP_DIR.exists():
            shutil.rmtree(APP_DIR)
        staging.replace(APP_DIR)
    finally:
        if tmp_path.exists():
            tmp_path.unlink()

run([sys.executable, "-m", "pip", "install", "-q", *APP_SERVER_PACKAGES])
install_release_app()

vllm_ver = subprocess.check_output([sys.executable, "-c", "import vllm; print(vllm.__version__)"], text=True).strip()
print(f"Engine: vLLM {vllm_ver} | app: {RELEASE_TAG} ({APP_SHA256[:12]})")
print("[OK] vLLM engine and Web Studio bundle installed.")

### 3. 下载 Qwen3.8-27B 模型
- **功能**：按显存下载模型至 `/content/models`：80GB 及以上为 `Qwen/Qwen3.8-27B` BF16（约 55 GB），40GB 为 `Qwen/Qwen3.8-27B-FP8`（约 28 GB）。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 3. Download & Verify Qwen3.8-27B Weights {display-mode: "form"}
import os, shutil, time, torch
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")
os.environ.setdefault("HF_XET_CHUNK_CACHE_SIZE_BYTES", "0")
from huggingface_hub import snapshot_download

MODEL_SPECS = {
    "BF16": {
        "repo_id": "Qwen/Qwen3.8-27B",
        "revision": "1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0",
        "local_dir": Path("/content/models/Qwen3.8-27B"),
        "min_free_gb": 58,
    },
    "FP8": {
        "repo_id": "Qwen/Qwen3.8-27B-FP8",
        "revision": "017b9c7af6b5689d5dd426a76e0bc077eb5ca20a",
        "local_dir": Path("/content/models/Qwen3.8-27B-FP8"),
        "min_free_gb": 31,
    },
}

def ensure_not_symlink(path: Path) -> None:
    for p in [path, *path.parents]:
        if p == Path("/content"):
            break
        if p.is_symlink():
            raise RuntimeError(f"Refusing symlink model path: {p}")

vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
precision = "BF16" if vram_gib >= 70 else "FP8"
spec = MODEL_SPECS[precision]
target_dir = spec["local_dir"]
ensure_not_symlink(target_dir)
target_dir.parent.mkdir(parents=True, exist_ok=True)

free_gb = shutil.disk_usage(target_dir.parent).free / 1024**3
if not (target_dir / "config.json").is_file() and free_gb < spec["min_free_gb"]:
    raise RuntimeError(f"Insufficient disk space: {spec['repo_id']} requires {spec['min_free_gb']} GB, available {free_gb:.1f} GB.")

print(f"Downloading: {spec['repo_id']} ({precision}, VRAM {vram_gib:.1f} GiB, commit {spec['revision'][:12]})")
started = time.time()
snapshot_download(
    repo_id=spec["repo_id"],
    revision=spec["revision"],
    local_dir=str(target_dir),
    token=False,
)
if target_dir.is_symlink() or (target_dir / "config.json").is_symlink():
    raise RuntimeError("Invalid model directory: symlink detected.")

SELECTED_QWEN38_DIR = str(target_dir)
marker = Path("/content/models/.selected_qwen38_dir")
if not marker.is_symlink():
    marker.write_text(SELECTED_QWEN38_DIR, encoding="utf-8")
print(f"[OK] Model ready: {target_dir} ({time.time() - started:.0f}s)")

### 4. 启动多模态对话 Web 工作台（Colab 原生代理）
- **功能**：在本地回环端口启动内部 `vllm serve`（`127.0.0.1:8189`，`262144` 上下文，MTP 投机采样）与 Web Studio 后端（`127.0.0.1:8188`），并通过 **Google Colab 原生端口代理** 打开网页对话工作台。
- **访问说明**：仅限当前登录的 Google 账号通过浏览器访问，无需密码或第三方隧道，不开放外部 `/v1/*` API 网关；实例停止后临时链接失效。
- **使用**：请在浏览器打开本 Colab 笔记本并直接运行下方单元格，完成后点击输出的链接打开网页工作台。

In [ ]:
# @title 4. Launch Web Studio {display-mode: "form"}
import json, os, subprocess, sys, time, urllib.request
from pathlib import Path
from urllib.parse import urlsplit

FEATURE_NAME = "qwen3.8-27b"
VLLM_PORT = 8189
VLLM_BASE = f"http://127.0.0.1:{VLLM_PORT}"
VLLM_LOG = Path("/content/vllm-worker.log")
VLLM_LAUNCH_CONFIG = Path("/content/vllm-worker.launch.json")
VLLM_PROCESS_PATTERN = r"vllm serve|vllm\.entrypoints"

APP_DIR = Path("/content/app")
DATA_DIR = Path(f"/content/app_data/{FEATURE_NAME}")
UPLOADS_DIR = DATA_DIR / "uploads"
PORT = 8188
BASE = f"http://127.0.0.1:{PORT}"
APP_LOG = Path(f"/content/{FEATURE_NAME}-web.log")
APP_LAUNCH_CONFIG = Path(f"/content/{FEATURE_NAME}.launch.json")
APP_PROCESS_PATTERN = r"backend/run_server\.py|backend\.run_server"

def read_text(path):
    return path.read_text(errors="replace") if path.exists() else ""

def tail(path, size=8000):
    return read_text(path)[-size:]

def rotate_log(path):
    if path.is_file() and path.stat().st_size:
        path.replace(path.with_name(path.name + ".1"))

def process_running(pattern):
    return subprocess.run(["pgrep", "-f", pattern], stdout=subprocess.DEVNULL).returncode == 0

def stop_processes(pattern, timeout):
    subprocess.run(["pkill", "-f", pattern], check=False)
    deadline = time.time() + timeout
    while process_running(pattern) and time.time() < deadline:
        time.sleep(2)
    if process_running(pattern):
        subprocess.run(["pkill", "-9", "-f", pattern], check=False)
        time.sleep(2)

def resolve_colab_proxy_url(port):
    try:
        from google.colab import output
    except ImportError:
        return f"http://127.0.0.1:{int(port)}", "local"
    try:
        proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({int(port)})")
    except Exception as exc:
        raise RuntimeError(
            "Failed to resolve Colab native proxy URL. Run Step 4 inside a browser Google Colab notebook session."
        ) from exc
    if not isinstance(proxy_url, str) or not proxy_url.strip().startswith("https://"):
        raise RuntimeError(
            f"Colab did not return a valid HTTPS proxy URL ({proxy_url!r}). Open the notebook in your browser and re-run Step 4."
        )
    return proxy_url.strip().rstrip("/"), "colab"

def open_colab_proxy(port, proxy_url, access_mode):
    if access_mode == "colab":
        from google.colab import output
        output.serve_kernel_port_as_window(int(port))
        print(f"[OK] Web Studio (Colab Native Proxy): {proxy_url}")
        print("Access is restricted to your signed-in Google account and expires when this runtime stops.")
    else:
        print(f"[OK] Web Studio (Local): {proxy_url}")

if APP_DIR.is_symlink() or not (APP_DIR / "backend/run_server.py").is_file():
    raise RuntimeError("App directory /content/app not found; please run Step 2 first.")

marker = Path("/content/models/.selected_qwen38_dir")
model_dir = (
    globals().get("SELECTED_QWEN38_DIR")
    or (marker.read_text(encoding="utf-8").strip() if marker.is_file() and not marker.is_symlink() else "")
    or ("/content/models/Qwen3.8-27B" if Path("/content/models/Qwen3.8-27B/config.json").is_file() else "/content/models/Qwen3.8-27B-FP8")
)
if Path(model_dir).is_symlink() or not (Path(model_dir) / "config.json").is_file():
    raise RuntimeError(f"Model directory not found: {model_dir}; please run Step 3 first.")

if DATA_DIR.is_symlink() or UPLOADS_DIR.is_symlink():
    raise RuntimeError("Symlink detected in data directory; aborting launch.")
UPLOADS_DIR.mkdir(parents=True, exist_ok=True)

# 1. Start internal vLLM engine (127.0.0.1:8189, loopback only)
def vllm_ready(timeout=5):
    try:
        with urllib.request.urlopen(VLLM_BASE + "/health", timeout=timeout) as r:
            return r.status == 200
    except Exception:
        return False

vllm_cmd = [
    "vllm", "serve", model_dir,
    "--served-model-name", "Qwen/Qwen3.8-27B", "qwen3.8-27b",
    "--host", "127.0.0.1",
    "--port", str(VLLM_PORT),
    "--max-model-len", "262144",
    # Single-GPU interactive use: avoid the throughput default exceeding hybrid-model cache blocks.
    "--max-num-seqs", "8",
    "--gpu-memory-utilization", "0.93",
    "--enable-prefix-caching",
    "--reasoning-parser", "qwen3",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "qwen3_coder",
    "--allowed-local-media-path", str(UPLOADS_DIR),
    "--trust-remote-code",
    "--speculative-config", '{"method":"mtp","num_speculative_tokens":3}',
]

vllm_cfg_str = json.dumps(vllm_cmd)
vllm_is_up = vllm_ready() or (process_running(VLLM_PROCESS_PATTERN) and vllm_ready(timeout=30))
if vllm_is_up and read_text(VLLM_LAUNCH_CONFIG) != vllm_cfg_str:
    print("vLLM launch configuration changed; restarting engine...")
    stop_processes(VLLM_PROCESS_PATTERN, 120)
    vllm_is_up = False

if vllm_is_up:
    print("[OK] Internal vLLM engine already running (127.0.0.1:8189).")
else:
    stop_processes(VLLM_PROCESS_PATTERN, 120)
    rotate_log(VLLM_LOG)
    with VLLM_LOG.open("w") as log:
        vllm_proc = subprocess.Popen(
            vllm_cmd,
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    VLLM_LAUNCH_CONFIG.write_text(vllm_cfg_str, encoding="utf-8")
    print("Loading 27B weights and initializing vLLM engine (256K context, max_num_seqs=8)...")
    deadline = time.time() + 900
    while time.time() < deadline:
        if vllm_proc.poll() is not None:
            print(tail(VLLM_LOG))
            raise RuntimeError(f"vLLM failed to start (exit code {vllm_proc.returncode}); see logs above.")
        if vllm_ready():
            print("[OK] Internal vLLM engine ready (127.0.0.1:8189).")
            break
        time.sleep(3)
    else:
        print(tail(VLLM_LOG))
        raise TimeoutError("vLLM did not become ready within 15 minutes; see logs above.")

# 2. Start FastAPI Web Playground server (127.0.0.1:8188, loopback only)
def app_ready(timeout=5):
    try:
        with urllib.request.urlopen(BASE + "/api/v1/health", timeout=timeout) as r:
            return r.status == 200
    except Exception:
        return False

proxy_url, access_mode = resolve_colab_proxy_url(PORT)
proxy_origin = f"{urlsplit(proxy_url).scheme}://{urlsplit(proxy_url).netloc}" if access_mode == "colab" else ""

env_overrides = {
    "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1",
    "COLAB_FEATURE": FEATURE_NAME,
    "COLAB_HOST": "127.0.0.1",
    "COLAB_PORT": str(PORT),
    "COLAB_DATA_DIR": str(DATA_DIR),
    "COLAB_DIST_DIR": str(APP_DIR / "dist" / FEATURE_NAME),
    "COLAB_PROXY_ORIGIN": proxy_origin,
    "VLLM_BASE_URL": VLLM_BASE,
    "VLLM_ALLOW_LOCAL_MEDIA": "1",
}

app_cmd = [
    sys.executable,
    str(APP_DIR / "backend/run_server.py"),
    "--feature",
    FEATURE_NAME,
    "--host",
    "127.0.0.1",
    "--port",
    str(PORT),
    "--dist-dir",
    str(APP_DIR / "dist" / FEATURE_NAME),
]

app_fingerprint = json.dumps({
    "cmd": app_cmd,
    "release": read_text(APP_DIR / ".release_sha256").strip(),
    "env": env_overrides,
}, sort_keys=True)
app_is_up = app_ready() or (process_running(APP_PROCESS_PATTERN) and app_ready(timeout=15))
if app_is_up and read_text(APP_LAUNCH_CONFIG) != app_fingerprint:
    print("Web Studio configuration changed; restarting...")
    stop_processes(APP_PROCESS_PATTERN, 30)
    app_is_up = False

if app_is_up:
    print("[OK] Web Studio already running (127.0.0.1:8188).")
else:
    stop_processes(APP_PROCESS_PATTERN, 30)
    rotate_log(APP_LOG)
    child_env = {**os.environ, **env_overrides}
    with APP_LOG.open("w") as log:
        app_proc = subprocess.Popen(
            app_cmd,
            cwd=APP_DIR,
            env=child_env,
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    APP_LAUNCH_CONFIG.write_text(app_fingerprint, encoding="utf-8")
    print("Starting Web Studio...")
    deadline = time.time() + 120
    while time.time() < deadline:
        if app_proc.poll() is not None:
            print(tail(APP_LOG))
            raise RuntimeError(f"Web Studio failed to start (exit code {app_proc.returncode}); see logs above.")
        if app_ready():
            print("[OK] Web Studio started.")
            break
        time.sleep(1.5)
    else:
        print(tail(APP_LOG))
        raise TimeoutError("Web Studio did not become ready within 2 minutes; check logs.")

# 3. Expose via Google Colab native proxy & persist live URL metadata
if not DATA_DIR.is_symlink():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    proxy_info_path = DATA_DIR / "proxy_info.json"
    if not proxy_info_path.is_symlink():
        proxy_info_path.write_text(
            json.dumps({"public_url": proxy_url, "access_mode": access_mode}, ensure_ascii=False),
            encoding="utf-8",
        )
open_colab_proxy(PORT, proxy_url, access_mode)